In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.8 MB/s eta 0:00:00


In [2]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
ret,frame = cap.read()
ret2, frame2 = cap.read()

In [4]:
def get_centroids(frame):
  result = model.predict(frame)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  centroid=[]

  for box,c in zip(boxes,conf):
    if c < 0.5:
      continue
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy))

  return centroid

In [5]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)


0: 384x640 11 persons, 319.4ms
Speed: 17.8ms preprocess, 319.4ms inference, 26.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 143.1ms
Speed: 3.9ms preprocess, 143.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)


In [6]:
players_position = {}
next_id = 0
for c in centroid_frame1:
  players_position[next_id] = [c]
  next_id +=1
print(players_position)

{0: [(np.float32(403.42545), np.float32(304.4803))], 1: [(np.float32(99.70708), np.float32(275.8715))], 2: [(np.float32(589.9822), np.float32(353.16367))], 3: [(np.float32(711.9473), np.float32(237.6838))], 4: [(np.float32(146.1174), np.float32(227.2714))], 5: [(np.float32(555.9518), np.float32(242.2549))], 6: [(np.float32(593.5323), np.float32(291.34247))], 7: [(np.float32(403.35294), np.float32(258.49933))]}


In [7]:
import math
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")   # reopen from frame 0 — the old cap object is exhausted
frame_count = 0
max_frames = 50   # quick test limit — remove once logic is confirmed correct
max_distance = 50

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)
    for c in centroid:
        best_id = None
        best_distance = float("inf")
        for pid, pos in players_position.items():
          d = math.dist(c, pos[-1])
          if d < best_distance:
              best_id = pid
              best_distance = d
        if best_distance < max_distance:
            players_position[best_id].append(c)   # adds to the list
        else:
            players_position[next_id] = [c]
            next_id += 1

print(players_position)
print(next_id)


0: 384x640 11 persons, 126.9ms
Speed: 3.4ms preprocess, 126.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 121.1ms
Speed: 4.9ms preprocess, 121.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 130.0ms
Speed: 2.6ms preprocess, 130.0ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 125.0ms
Speed: 4.5ms preprocess, 125.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 130.9ms
Speed: 6.7ms preprocess, 130.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 122.9ms
Speed: 2.7ms preprocess, 122.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 145.4ms
Speed: 3.4ms preprocess, 145.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 persons, 122.2ms
Speed: 2.6ms preprocess, 122.2ms inference, 0.9ms postproc

In [8]:
player_distances = {}

for pid, history in players_position.items():
    total = 0
    for i in range(len(history) - 1):
        x1, y1 = history[i]
        x2, y2 = history[i+1]
        d = ((x2-x1)**2 + (y2-y1)**2)**0.5
        total += d
    player_distances[pid] = total

print(player_distances)

{0: np.float32(50.36294), 1: np.float32(195.61069), 2: np.float32(58.813572), 3: np.float32(56.40588), 4: np.float32(76.02489), 5: np.float32(42.017612), 6: np.float32(41.152836), 7: np.float32(128.33417), 8: np.float32(41.621933), 9: 0, 10: np.float32(12.96672)}


In [9]:
def get_jersey_crop(frame, box):
    # cast to int since slicing needs whole numbers, not the floats YOLO gives
    x1, y1, x2, y2 = map(int, box)

    height = y2 - y1
    # only take top 35% of box height to isolate jersey, skip shorts/legs
    new_y2 = int(y1 + (0.35 * height))

    # rows (y) first, then columns (x) — standard image slicing order
    return frame[y1:new_y2, x1:x2]

In [10]:

def get_avg_color(crop):
  return crop.mean(axis = (0,1))

In [11]:
avg_colors = []
result = model.predict(frame)
boxes = result[0].boxes.xyxy.cpu().numpy()
for box in boxes:
  crop = get_jersey_crop(frame, box)
  avg_color = get_avg_color(crop)
  avg_colors.append(avg_color)

print(len(avg_colors))
print(avg_colors[0])


0: 384x640 12 persons, 128.4ms
Speed: 3.1ms preprocess, 128.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
12
[     75.588      135.89      121.67]


In [12]:
data = np.array(avg_colors, dtype=np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
compactness, labels, centers = cv2.kmeans(data, 2, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

print(compactness, labels, centers)

4211.715896606445 [[0]
 [1]
 [1]
 [1]
 [1]
 [0]
 [0]
 [0]
 [1]
 [1]
 [0]
 [0]] [[     83.438      120.89      123.11]
 [     109.78      159.15      142.31]]


In [13]:
player_team = {}
for i,box in enumerate(boxes):
  x1, y1, x2, y2 = box
  cx = (x1+x2)/2
  cy= (y1+y2)/2

  best_id = None
  best_distance = float("inf")
  for pid,pos in players_position.items():
    d = math.dist((cx,cy), pos[-1])
    if d<best_distance:
      best_distance = d
      best_id = pid

  if best_id not in player_team :
    player_team[best_id] = labels[i]

print(player_team)

{1: array([0], dtype=int32), 2: array([1], dtype=int32), 0: array([1], dtype=int32), 5: array([1], dtype=int32), 6: array([1], dtype=int32), 10: array([0], dtype=int32), 4: array([0], dtype=int32), 7: array([0], dtype=int32), 8: array([1], dtype=int32)}


In [20]:
team_distance = {}
for pid, dist in player_distances.items():
  if pid not in player_team:
    continue
  team = int(player_team[pid])
  if team not in team_distance:
    team_distance[team]= 0
  team_distance[team] += dist

print(team_distance)


{1: np.float32(233.96889), 0: np.float32(412.93643)}


/tmp/ipykernel_3088/3581967056.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  team = int(player_team[pid])
